# Gibbs Sampling and the Metropolis–Hastings Algorithm

**Markov Chain Monte Carlo (MCMC)** methods generate samples from a target distribution $\pi(x) \propto e^{-E(x)/\varepsilon}$ by constructing an ergodic Markov chain whose stationary distribution is $\pi$. When direct sampling is intractable, MCMC provides the gold standard for Bayesian inference, statistical physics, and optimization.

## Metropolis–Hastings algorithm

Starting from a current state $x^k$, the MH algorithm:
1. **Propose** $y \sim q(\cdot | x^k)$ (e.g., Gaussian random walk: $y = x^k + \sigma Z$, $Z \sim \mathcal{N}(0,I)$).
2. **Accept/reject** with acceptance probability
$$
\alpha(x^k, y) = \min\!\left(1,\, \frac{\pi(y)\,q(x^k|y)}{\pi(x^k)\,q(y|x^k)}\right) = \min\!\left(1,\, e^{(E(x^k)-E(y))/\varepsilon}\right)
$$
for symmetric proposals. The chain satisfies **detailed balance** and converges to $\pi$.

## Gibbs sampling

In high dimensions, **Gibbs sampling** updates one coordinate at a time by sampling from the full conditional $\pi(x_i | x_{-i})$. For a 2D distribution, the chain alternates:
$$
x_1^{k+1} \sim \pi(x_1 | x_2^k), \qquad x_2^{k+1} \sim \pi(x_2 | x_1^{k+1}).
$$
Gibbs is a special case of MH with acceptance rate 1.

## Temperature and mixing

The parameter $\varepsilon > 0$ acts as a **temperature**:
- $\varepsilon \to 0$: samples concentrate near the modes (low-temperature, greedy).
- $\varepsilon \to \infty$: samples spread uniformly (high-temperature, no preference).
The **mixing time** — how many steps until the chain forgets its starting point — grows as $\varepsilon \to 0$ for multimodal distributions.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## Defining the target distribution

We work on $[0,1]^2$ with an energy $E(x)$ composed of three Gaussian wells (local minima). The Gibbs distribution is $\pi(x) \propto e^{-E(x)/\varepsilon}$.

In [ ]:
n_disp = 300
t = np.linspace(0, 1, n_disp)
Y, X = np.meshgrid(t, t)
Z = X + 1j * Y

centers = [0.3 + 0.7j, 0.6 + 0.25j, 0.65 + 0.75j]
scales  = [0.10, 0.13, 0.08]
weights = [0.8, 0.9, 1.0]

def energy(z):
    E = np.ones_like(np.abs(z), dtype=float)
    for c, s, w in zip(centers, scales, weights):
        E -= w * np.exp(-np.abs(z - c)**2 / (2 * s**2))
    return E

E_grid = energy(Z)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.contourf(t, t, E_grid.T, levels=14, cmap='viridis')
ax.contour(t, t, E_grid.T, levels=14, colors='k', linewidths=0.5, alpha=0.5)
for c in centers:
    ax.plot(c.real, c.imag, 'w*', ms=10)
ax.set_title('Energy landscape $E(x)$'); ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Metropolis–Hastings sampler

We run MH with a Gaussian random-walk proposal of standard deviation $\sigma$. Multiple particles are evolved in parallel to illustrate the ergodic exploration.

In [ ]:
def metropolis_hastings(n_steps, n_particles=20, sigma=0.02, eps=0.1, seed=42):
    rng = np.random.default_rng(seed)
    z = 0.5 + 0.5j + (rng.standard_normal(n_particles) + 1j * rng.standard_normal(n_particles)) * 0.01
    traj = [z.copy()]
    E_curr = energy(z)
    n_accept = 0

    for _ in range(n_steps):
        z_prop = z + (rng.standard_normal(n_particles) + 1j * rng.standard_normal(n_particles)) * sigma
        # Keep in [0,1]^2
        z_prop = np.clip(z_prop.real, 0, 1) + 1j * np.clip(z_prop.imag, 0, 1)
        E_prop = energy(z_prop)
        log_alpha = (E_curr - E_prop) / eps
        accept = np.log(rng.uniform(size=n_particles)) < log_alpha
        z = np.where(accept, z_prop, z)
        E_curr = np.where(accept, E_prop, E_curr)
        n_accept += accept.sum()
        traj.append(z.copy())

    return np.array(traj), n_accept / (n_steps * n_particles)

traj, acc_rate = metropolis_hastings(n_steps=500, n_particles=15, sigma=0.02, eps=0.08)
print(f'Acceptance rate: {acc_rate:.2%}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, step, title in zip(axes, [10, 100, 500], ['Early (t=10)', 'Mid (t=100)', 'Late (t=500)']):
    ax.contourf(t, t, E_grid.T, levels=14, cmap='viridis', alpha=0.7)
    tail = min(step, 50)
    for p in range(traj.shape[1]):
        seg = traj[max(0, step-tail):step+1, p]
        ax.plot(seg.real, seg.imag, '-', lw=1, alpha=0.6)
        ax.plot(traj[step, p].real, traj[step, p].imag, 'w.', ms=6)
    ax.set_title(title, fontsize=10); ax.axis('off')

fig.suptitle('MH chain exploration of a 3-well energy', y=1.02)
plt.tight_layout(); plt.show()

## Effect of temperature $\varepsilon$

At low temperature the chain concentrates in the deepest well; at high temperature it explores more uniformly. We compare the stationary histograms for four temperatures.

In [ ]:
eps_values = [0.02, 0.06, 0.15, 0.5]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for col, eps in enumerate(eps_values):
    traj_e, _ = metropolis_hastings(n_steps=2000, n_particles=30, sigma=0.02, eps=eps, seed=col)
    samples = traj_e[500:].reshape(-1)  # discard burn-in

    # Trajectories
    ax = axes[0, col]
    ax.contourf(t, t, E_grid.T, levels=12, cmap='viridis', alpha=0.6)
    for p in range(min(5, traj_e.shape[1])):
        ax.plot(traj_e[-200:, p].real, traj_e[-200:, p].imag, '-', lw=0.8, alpha=0.7)
    ax.set_title(fr'$\varepsilon = {eps}$', fontsize=9); ax.axis('off')

    # 2D histogram
    ax2 = axes[1, col]
    ax2.hist2d(samples.real, samples.imag, bins=40, range=[[0,1],[0,1]], cmap='hot')
    for c in centers:
        ax2.plot(c.real, c.imag, 'c*', ms=8)
    ax2.set_aspect('equal'); ax2.axis('off')

axes[0,0].set_ylabel('trajectories', fontsize=9)
axes[1,0].set_ylabel('sample histogram', fontsize=9)
fig.suptitle('MH sampling at different temperatures: low = concentrated, high = diffuse', y=1.02)
plt.tight_layout(); plt.show()

## Autocorrelation and mixing time

The **autocorrelation** $\rho(\tau) = \text{Corr}(x^k, x^{k+\tau})$ measures how long the chain remembers its past. The **integrated autocorrelation time** $\tau_{\text{int}} = \frac{1}{2} + \sum_{\tau=1}^\infty \rho(\tau)$ quantifies the effective sample size per step.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(eps_values)))

for eps, col in zip(eps_values, cols):
    traj_e, _ = metropolis_hastings(n_steps=3000, n_particles=1, sigma=0.02, eps=eps, seed=7)
    x_chain = traj_e[:, 0].real
    x_chain -= x_chain.mean()
    lag_max = 150
    acf = np.array([np.corrcoef(x_chain[:-lag], x_chain[lag:])[0,1] if lag > 0 else 1.0
                    for lag in range(lag_max)])
    ax.plot(acf, lw=2, color=col, label=fr'$\varepsilon={eps}$')

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('lag $\\tau$'); ax.set_ylabel('autocorrelation $\\rho(\\tau)$')
ax.set_title('Chain autocorrelation: higher $\\varepsilon$ → faster mixing')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Interactive: tune temperature and step size

In [ ]:
def show_mh(log_eps=-1.5, sigma=0.025, n_steps=800):
    eps = 10**log_eps
    traj_i, acc = metropolis_hastings(n_steps, n_particles=20, sigma=sigma, eps=eps)
    samples = traj_i[n_steps//4:].reshape(-1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].contourf(t, t, E_grid.T, levels=12, cmap='viridis', alpha=0.7)
    for p in range(min(6, traj_i.shape[1])):
        seg = traj_i[-100:, p]
        axes[0].plot(seg.real, seg.imag, '-', lw=1, alpha=0.7)
    axes[0].set_title(fr'Trajectories ($\varepsilon={eps:.3f}$, acc={acc:.0%})')
    axes[0].axis('off')
    axes[1].hist2d(samples.real, samples.imag, bins=40, range=[[0,1],[0,1]], cmap='hot')
    for c in centers: axes[1].plot(c.real, c.imag, 'c*', ms=10)
    axes[1].set_title('Sample histogram'); axes[1].set_aspect('equal'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

interact(show_mh,
         log_eps=FloatSlider(value=-1.5, min=-3.0, max=0.0, step=0.25, description='$\\log_{10}\\varepsilon$'),
         sigma=FloatSlider(value=0.025, min=0.005, max=0.1, step=0.005, description='$\\sigma$'),
         n_steps=IntSlider(value=800, min=200, max=3000, step=200, description='steps'));

## Bibliographical resources

- Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H. and Teller, E. (1953). Equation of state calculations by fast computing machines. *The Journal of Chemical Physics*, 21(6), 1087–1092.
- Hastings, W. K. (1970). Monte Carlo sampling methods using Markov chains and their applications. *Biometrika*, 57(1), 97–109.
- Geman, S. and Geman, D. (1984). Stochastic relaxation, Gibbs distributions, and the Bayesian restoration of images. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 6(6), 721–741.
- Robert, C. P. and Casella, G. (2004). *Monte Carlo Statistical Methods* (2nd ed.). Springer.
- Brooks, S., Gelman, A., Jones, G. and Meng, X.-L. (Eds.) (2011). *Handbook of Markov Chain Monte Carlo*. CRC Press.